In [ ]:
import pickle
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

In [ ]:
annots = pd.read_csv('/mnt/nfs/home/jasonyeung/PDAC/annotations/annotations_2023-08-17_FINAL.csv', low_memory = False)
annots.head()

In [ ]:
annots['cellType'].unique()

In [ ]:
annots["confirmed_cell"] = np.where(annots["cellType"] == "Junk", False, 
                                    (np.where(annots["cellType"] == "Other", False, True)))

clean_annots = annots[annots.confirmed_cell != False]
clean_annots = clean_annots.rename({'cellType': 'cluster'}, axis=1)

In [ ]:
key = [x for x in np.unique(clean_annots["PointNum"])]
df_list = [clean_annots[clean_annots["PointNum"] == x] for x in key]
df_dict = dict(zip(key, df_list))
clean_annots.sort_values("PointNum")

In [ ]:
import functools
import spatial_lda
import seaborn as sns
from spatial_lda.featurization import featurize_samples
from spatial_lda.featurization import neighborhood_to_cluster
from spatial_lda.featurization import make_merged_difference_matrices

In [ ]:
blue = sns.color_palette()[0]
green = sns.color_palette()[1]
red = sns.color_palette()[2]
sns.set_context("paper",font_scale=1)
sns.set_style('white')
sns.set_context("notebook", font_scale=1.0, rc={"lines.linewidth": 1.0})

In [ ]:
pdac_features = featurize_samples(df_dict, neighborhood_to_cluster, radius = 100, 
                                  is_anchor_col = "confirmed_cell", x_col = 'X_cent', y_col = 'Y_cent', 
                                  n_processes = 20, include_anchors = True)

In [ ]:
pdac_difference_matrices = make_merged_difference_matrices(pdac_features, df_dict, x_col = 'X_cent', y_col = 'Y_cent')

In [ ]:
from spatial_lda.visualization import plot_adjacency_graph
from spatial_lda.visualization import plot_samples_in_a_row
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
all_sample_idxs = pdac_features.index.map(lambda x: x[0])

_sets = train_test_split(pdac_features, test_size = 0.1, stratify = all_sample_idxs)

In [ ]:
train_pdac_features, test_pdac_features = _sets

train_difference_matrices = make_merged_difference_matrices(train_pdac_features, df_dict, x_col = 'X_cent', y_col = 'Y_cent')

In [ ]:
import os
from spatial_lda.model import order_topics_consistently

np.random.seed(2023)
random.seed(2023)

path_to_train_model = 'models'

spatial_lda_models = {}
difference_penalty = 0.25

N_TOPICS_LIST = [6, 7, 8, 10, 11]

for n in N_TOPICS_LIST:
    path_to_train_model = '_'.join((f'/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/models/PDAC',f'penalty={difference_penalty}', f'topics={n}',f'trainfrac=0.9')) + '.pkl'
    print(f'Running n_topics = {n}, d = {difference_penalty}\n')
    spatial_lda_model = spatial_lda.model.train(sample_features = train_pdac_features,
                                               difference_matrices = train_difference_matrices,
                                               difference_penalty = difference_penalty,
                                               n_topics = n,
                                               n_parallel_processes = 20,
                                               verbosity = 1,
                                               admm_rho = 0.1,
                                               primal_dual_mu = 1e+5)
    spatial_lda_models[n] = spatial_lda_model
    with open(path_to_train_model, 'wb') as f:
        pickle.dump(spatial_lda_model, f)

order_topics_consistently(spatial_lda_models.values())

In [ ]:
from spatial_lda.visualization import plot_topics_heatmap
normalizer = lambda ar: (ar/np.sum(ar, axis=1, keepdims=True)).T
# idcs_take = [0,1,2,3,4,5,6,7,8,10,11,12,13,14]

def _standardize_topics(topics):
    topics = topics.T
    topics = topics - topics.mean(axis=1, keepdims=True)
    topics = topics / topics.std(axis=1, keepdims=True)
    return topics

def plot_topics_heatmap(topics, features, normalizer=None):
    n_topics = topics.shape[0]
    if normalizer is not None:
        topics = normalizer(topics)
    else:
        topics = _standardize_topics(topics)

    topics = pd.DataFrame(topics, index=features,
                          columns=['Topic %d' % x for x in range(n_topics)])
    sns.heatmap(topics, square=True, cmap='vlag', center = 0, vmin = -2, vmax = 2)

with open('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/models/PDAC_penalty=0.25_topics=6_trainfrac=0.9.pkl', 'rb') as pickle_file:
    lda_6 = pickle.load(pickle_file)

with open('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/models/PDAC_penalty=0.25_topics=7_trainfrac=0.9.pkl', 'rb') as pickle_file:
    lda_7 = pickle.load(pickle_file)

with open('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/models/PDAC_penalty=0.25_topics=8_trainfrac=0.9.pkl', 'rb') as pickle_file:
    lda_8 = pickle.load(pickle_file)

with open('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/models/PDAC_penalty=0.25_topics=10_trainfrac=0.9.pkl', 'rb') as pickle_file:
    lda_10 = pickle.load(pickle_file)

with open('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/models/PDAC_penalty=0.25_topics=11_trainfrac=0.9.pkl', 'rb') as pickle_file:
    lda_11 = pickle.load(pickle_file)

models = [lda_6, lda_7, lda_8, lda_10, lda_11]
num_models = len(models)

heatmap_width = 5
heatmap_height = 7

plt.figure(figsize=((heatmap_width * num_models), heatmap_height))
for i, model in enumerate(models):
    plt.subplot(1, num_models, i+1)
    plot_topics_heatmap(model.components_[:, :], pdac_features.columns)

plt.tight_layout(pad = 3.0)
plt.show()

In [ ]:
import os
from spatial_lda.model import order_topics_consistently

complete_lda = spatial_lda.model.infer(lda_8.components_, pdac_features, pdac_difference_matrices, difference_penalty = 0.25, n_parallel_processes = 20)

In [ ]:
complete_lda.topic_weights.to_csv('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/topic_weights_n8.csv')

In [ ]:
plt.figure(figsize = [12, 12])
plot_topics_heatmap(complete_lda.components_[:, :], pdac_features.columns)
plt.savefig("/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/pdac_lda_n8.svg")
plt.savefig("/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/pdac_lda_n8.pdf")

In [ ]:
topic_weights = complete_lda.topic_weights.copy()
topic_weights.index.name = "cell_idx"
topic_weights.reset_index(inplace = True)
topic_weights["PointNum"], topic_weights["idx"] = topic_weights.cell_idx.str
topic_weights.set_index("idx", inplace=True)
topic_weights.index.name = None

topic_weights

In [ ]:
joined = pd.merge(clean_annots, topic_weights, left_index = True, right_index = True)
joined['PointNum_x'].equals(joined['PointNum_y'])

In [ ]:
joined.drop(columns=["cell_idx", "PointNum_y", "confirmed_cell"], inplace = True)
joined.rename(columns={"PointNum_x": "PointNum"}, inplace = True)
joined["Topic"] = joined.filter(regex="^Topic").idxmax(axis=1)
joined

In [ ]:
joined.to_csv('/mnt/nfs/home/jasonyeung/PDAC/SPATIAL_LDA/full_topics.csv', sep = ',')